# Run ERM (SGNM equivariant model) on a GPU

**First: Runtime → Change runtime type → T4 GPU.** Then run the cells in order.

Note: cell 1b builds flash-eq's CUDA kernel from source and then asks you to
**restart the session** once (prebuilt wheels often don't match Colab's
Python/CUDA, so the kernel must be compiled here). After the restart, skip to
cell 2 and run the rest.

Output: `results/erm_profiles.csv` (downloaded at the end). Back on your Mac,
drop it in `results/` and run `python3 scripts/ingest_erm.py`.

In [ ]:
# 1a. Base deps
!apt-get -qq install -y gperf >/dev/null   # ciffy C extension needs gperf
!pip -q install numpy pandas scipy h5py tqdm

# dlu (pure python; dist name is 'dlu-torch', imports as 'dlu')
!pip -q install git+https://github.com/hmblair/dlu.git

# ciffy: upstream pyproject has a DUPLICATE fallback_version that breaks the
# build (tomllib 'Cannot overwrite a value'). Clone, patch, then install.
!rm -rf /content/ciffy && git clone -q https://github.com/hmblair/ciffy.git /content/ciffy
import pathlib
_pp = pathlib.Path('/content/ciffy/pyproject.toml')
_pp.write_text(_pp.read_text().replace('fallback_version = "0.0.0"\n', '', 1))
!pip -q install /content/ciffy

# sgnm: its pyproject pins 'dlu' (the dist is 'dlu-torch'), so install --no-deps
!pip -q install --no-deps git+https://github.com/hmblair/sgnm.git
print('base deps installed')

In [ ]:
# 1b. flash-eq: BUILD THE CUDA KERNEL FROM SOURCE.
# Prebuilt wheels often don't match Colab's Python/torch-CUDA, leaving the
# '_block_diagonal_cuda' extension missing. Building here against Colab's CUDA
# (nvcc is present) is reliable. Takes ~1-2 min.
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
!pip -q uninstall -y flash-eq 2>/dev/null
!pip -q install --no-build-isolation --no-cache-dir git+https://github.com/hmblair/flash-eq.git
print('\n>>> Now: Runtime -> Restart session, then run cell 2 onward (skip cell 1).')

In [ ]:
# 2. (after restart) verify GPU + the flash-eq CUDA extension, then get data
import torch, ciffy, dlu, sgnm
from flash_eq import _block_diagonal_cuda   # errors if the kernel didn't build
from sgnm import EquivariantReactivityModel
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU'
print('imports + CUDA extension OK | GPU:', torch.cuda.get_device_name(0))

%cd /content
!rm -rf rna-shape-decoys && git clone -q https://github.com/rhiju/rna-shape-decoys.git
%cd rna-shape-decoys
!curl -sL -o equivariant-checkpoint.pth \
  https://github.com/hmblair/sgnm/releases/download/v2.0.2/equivariant-checkpoint.pth
!ls -lh equivariant-checkpoint.pth   # should be ~1 MB (not a few bytes)

In [ ]:
# 3. Sanity check: ciffy must type atoms correctly (column-aligned converter)
import sys; sys.path.insert(0, 'scripts')
import ciffy, numpy as np, tempfile
from pdb_to_cif import pdb_to_cif
with tempfile.NamedTemporaryFile(suffix='.cif') as tf:
    pdb_to_cif('data/farfar2/Mol9_reference_UtoG_buildloop.pdb', tf.name)
    p = ciffy.load(tf.name, backend='numpy')
a = np.asarray(p.atoms)
print('code-0 atoms (must be 0):', int((a == 0).sum()), '/', len(a))
assert (a == 0).sum() == 0, 'atom typing failed — check pdb_to_cif'

In [ ]:
# 4. Run ERM on all structures (writes results/erm_profiles.csv)
!python3 scripts/run_erm.py equivariant-checkpoint.pth

In [ ]:
# 5. Download results/erm_profiles.csv
from google.colab import files
files.download('results/erm_profiles.csv')